In [1]:
from load_ud_dataset import load_ud_dataset, UD_GERMAN_SPLITS

In [2]:
UD_GERMAN_GSD_SPLITS = {
    'train': 'de_gsd-ud-train.conllu',
    'dev': 'de_gsd-ud-dev.conllu',
    'test': 'de_gsd-ud-test.conllu',
}
ud_path = "../data/UD_German-GSD"

In [3]:
ud = load_ud_dataset(ud_path, splits_filemap=UD_GERMAN_SPLITS[0])

/opt/anaconda3/envs/morph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded train: 13813 sentences
Loaded dev: 799 sentences
Loaded test: 977 sentences


In [4]:
ud

DatasetDict({
    train: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 13813
    })
    dev: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 799
    })
    test: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 977
    })
})

In [5]:
from conllu import parse_incr
import pandas as pd

from ud_corruption_utils import (
    case_marker_lemma_for_head_id,
    classify_dative_type,
    is_target_dative_preposition,
    normalize_case_prep_lemma,
)

########################################################################
# CONFIG
########################################################################

DAT_POS = {"NOUN", "PROPN", "PRON"}

########################################################################
# HELPERS
########################################################################

def has_dative(feats):
    if feats is None:
        return False
    return feats.get("Case") == "Dat"


def get_token_by_id(sent, idx):
    for tok in sent:
        if tok["id"] == idx:
            return tok
    return None


def get_children(sent, head_id):
    return [tok for tok in sent if tok["head"] == head_id]


def collect_span(sent, head_id):
    """
    Collect a simple NP span around the dative head (no preposition).
    Case markers / prepositions are stored via get_case_marker() and get_preposition().
    """

    allowed = {
        "det",
        "amod",
        "nummod",
        "compound",
        "fixed",
        "flat",
    }

    span_tokens = []

    for tok in sent:
        if tok["id"] == head_id:
            span_tokens.append(tok)

        elif tok["head"] == head_id and tok["deprel"] in allowed:
            span_tokens.append(tok)

    span_tokens = sorted(span_tokens, key=lambda x: x["id"])

    return span_tokens


def get_case_marker(sent, head_id):
    """UD ``case`` dependent lemma (preposition or directional adverb, etc.)."""
    return case_marker_lemma_for_head_id(sent, head_id)


def get_preposition(sent, head_id):
    """Target preposition (aus/bei/…/in); None for nordwestlich, dank, …"""
    case_lemma = get_case_marker(sent, head_id)
    if not is_target_dative_preposition(case_lemma):
        return None
    return normalize_case_prep_lemma(case_lemma)


def classify_dative(token, case_lemma):
    type = classify_dative_type(token, case_lemma)
    if type == "comparative_case_dative":
        print("## comparative_case_dative")
    return type

In [6]:

########################################################################
# MAIN
########################################################################

splits = {
    'train': '../data/UD_German-GSD/de_gsd-ud-train.conllu',
    'dev': '../data/UD_German-GSD/de_gsd-ud-dev.conllu',
    'test': '../data/UD_German-GSD/de_gsd-ud-test.conllu',
}

all_dfs = []

for split in splits:
    rows = []

    with open(splits[split], "r", encoding="utf-8") as f:

        for sent in parse_incr(f):

            sent_id = sent.metadata.get("sent_id", "")
            text = sent.metadata.get("text", "")

            for tok in sent:

                if tok["upos"] not in DAT_POS:
                    continue

                if not has_dative(tok["feats"]):
                    continue

                span = collect_span(sent, tok["id"])

                span_text = " ".join(t["form"] for t in span)

                case_marker = get_case_marker(sent, tok["id"])
                prep = get_preposition(sent, tok["id"])

                gov = get_token_by_id(sent, tok["head"])

                if gov is None:
                    gov_form = "ROOT"
                    gov_upos = "ROOT"
                else:
                    gov_form = gov["form"]
                    gov_upos = gov["upos"]

                subtype = classify_dative(tok, case_marker)

                rows.append({
                    "sent_id": sent_id,
                    "sentence": text,

                    "span": span_text,
                    "head_form": tok["form"],
                    "head_lemma": tok["lemma"],
                    "head_upos": tok["upos"],

                    "case_marker": case_marker,
                    "preposition": prep,
                    "dative_type": subtype,

                    "relation": tok["deprel"],

                    "governor": gov_form,
                    "governor_upos": gov_upos,
                    "split": split,  # add split information for provenance
                })

    df = pd.DataFrame(rows)
    all_dfs.append(df)

# Combine into unified DataFrame
unified_df = pd.concat(all_dfs, ignore_index=True)

print(unified_df.head())

unified_df.to_csv("dative_spans/german_datives_all.csv", index=False)

## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
## comparative_case_dative
#

In [7]:
set(df.dative_type)

{'comparative_case_dative',
 'core_dative_argument',
 'dative_prep_an',
 'dative_prep_auf',
 'dative_prep_aus',
 'dative_prep_bei',
 'dative_prep_hinter',
 'dative_prep_in',
 'dative_prep_mit',
 'dative_prep_nach',
 'dative_prep_neben',
 'dative_prep_seit',
 'dative_prep_unter',
 'dative_prep_von',
 'dative_prep_vor',
 'dative_prep_zu',
 'dative_prep_zwischen',
 'dative_prep_über',
 'nominal_dative_modifier',
 'oblique_dative',
 'other_dative',
 'other_prepositional_dative'}

In [8]:
set(unified_df.preposition)

{None,
 'an',
 'auf',
 'aus',
 'bei',
 'hinter',
 'in',
 'mit',
 'nach',
 'neben',
 'seit',
 'unter',
 'von',
 'vor',
 'zu',
 'zwischen',
 'über'}

In [9]:
lookup_dict = pd.read_csv("german_ud_lookup_dictionary.csv")
lookup_dict.head()

/var/folders/cp/gk01f5vx6hsdhrv8bptt9l440000gn/T/ipykernel_46997/3561946569.py:1: DtypeWarning: Columns (4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup_dict = pd.read_csv("german_ud_lookup_dictionary.csv")


,Lemma,Case,Number,Gender,Degree,Upos,Inflection,Forms
0,A,Dat,Sing,Neut,NaN,PROPN,NaN,A
1,A,Dat,Sing,Fem,NaN,PROPN,NaN,A
2,A,Dat,Sing,Fem,NaN,NOUN,NaN,A
3,A,Acc,Sing,NaN,NaN,PROPN,NaN,A
4,A,Dat,Sing,Masc,NaN,PROPN,NaN,A


In [10]:
set(lookup_dict.Upos)

{'ADJ', 'ADP', 'ADV', 'DET', 'NOUN', 'NUM', 'PRON', 'PROPN', 'X'}

In [11]:
# how many rows has no Gender?
lookup_dict[lookup_dict.Gender.isna()]


,Lemma,Case,Number,Gender,Degree,Upos,Inflection,Forms
3,A,Acc,Sing,NaN,NaN,PROPN,NaN,A
8,A.V.G.,Dat,Sing,NaN,NaN,PROPN,NaN,A.V.G.
14,A36,Acc,Sing,NaN,NaN,PROPN,NaN,A36
20,ABB,Dat,Sing,NaN,NaN,PROPN,NaN,ABB
23,ABC,Dat,Sing,NaN,NaN,PROPN,NaN,ABC
...,...,...,...,...,...,...,...,...
121720,üblich,Dat,Plur,NaN,Pos,ADJ,NaN,üblichen
121730,üblich,Acc,Plur,NaN,Pos,ADJ,Strong,übliche
121734,übrig,Dat,Plur,NaN,Pos,ADJ,Weak,übrigen
121742,übrig,Dat,Sing,NaN,Pos,ADJ,Strong,übrigen


In [12]:
# how many rows has no Number?
lookup_dict[lookup_dict.Number.isna()]


,Lemma,Case,Number,Gender,Degree,Upos,Inflection,Forms
47,ACT,Acc,NaN,NaN,NaN,PROPN,NaN,ACT
59,ADSL,Acc,NaN,NaN,NaN,NOUN,NaN,ADSL
110,AFR,Dat,NaN,NaN,NaN,NOUN,NaN,AFR
117,AGBs,Dat,NaN,NaN,NaN,NOUN,NaN,AGBs
154,AIDS,Dat,NaN,NaN,NaN,NOUN,NaN,AIDS
...,...,...,...,...,...,...,...,...
121425,über,Dat,NaN,NaN,NaN,ADP,NaN,über
121610,überschreiben,Dat,NaN,NaN,Pos,ADJ,Weak,überschrieben
121676,überwiegend,Dat,NaN,NaN,Pos,ADJ,NaN,überwiegend
121711,üblich,Dat,NaN,NaN,Pos,ADJ,Weak,üblich


In [13]:
import csv
import pandas as pd

In [14]:
dative_spans = pd.read_csv("dative_spans/german_datives_all.csv")
set(dative_spans.preposition)

{'an',
 'auf',
 'aus',
 'bei',
 'hinter',
 'in',
 'mit',
 'nach',
 nan,
 'neben',
 'seit',
 'unter',
 'von',
 'vor',
 'zu',
 'zwischen',
 'über'}

In [15]:
# find the row with the preposition "nordwestlich"
dative_spans[dative_spans.preposition == "nordwestlich"]

,sent_id,sentence,span,head_form,head_lemma,head_upos,case_marker,preposition,dative_type,relation,governor,governor_upos,split
